## Randomly Initialized Transformer Architecture

##### Only Inference...

##### Used Tokenizer : BPE Tokenizer from tiktoken library

In [ ]:
import tiktoken
import numpy as np
import torch
from torch import nn

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

In [ ]:
def softmax(x, axis= 0):
    # subtract max for numerical stability

    # across rows
    x_shifted = x - np.max(x, axis = axis, keepdims=True)
    
    exp_x = np.exp(x_shifted)
    
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)

In [ ]:
def self_attention(query , key, value):
    # q, k, v are np arrays and are of same order
    # writing codea asuming they are horizontal matrices
    dk = key.shape[1]
    a = (query @ key.T) / np.sqrt(dk) # dot product but for a number of words will be orchestrsted by cross product

    a = softmax(a , axis = 1)
    
    b = a @ value
    
    return b
    

In [ ]:
class MultiHeadAttention:

    def __init__(self, embed_dim, num_heads, masked = False):
        self.embed_dim = embed_dim 
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.masked = masked

        self.WQ = [np.random.randn(embed_dim, self.head_dim) for _ in range(num_heads)]
        self.WK = [np.random.randn(embed_dim, self.head_dim) for _ in range(num_heads)]
        self.WV = [np.random.randn(embed_dim, self.head_dim) for _ in range(num_heads)]

        self.WO = np.random.randn(embed_dim, embed_dim)

    def self_attention(self,query , key, value):
        score = (query @ key.T) / np.sqrt(self.head_dim)
        if (self.masked):
            r,c = score.shape
            for i in range(r):
                for j in range(i+1, c):
                    score[i][j] = -np.inf



        attention = softmax(score , axis = 1)
        
        b = attention @ value
        return b
            
    
    def forward(self, query, key, value):
        heads =  []
        for i in range(self.num_heads):

            q = query @ self.WQ[i]
            k = key @ self.WK[i]
            v = value @ self.WV[i]
            head = self.self_attention(q, k, v)
            heads.append(head)

        context_awared = np.concatenate(heads, axis = 1)

        output = context_awared @ self.WO # without this step all the heads will be separate. But this combines the information of all the heads
        return output

positional encoder

In [ ]:
# based on formula

def positional_encoder_f(pos, embedding_dimension):

    if (embedding_dimension % 2 != 0 ):
        raise ValueError("Embedding dimension must be even.")
    encoded_pos = []    
    for i in range (int((embedding_dimension / 2))) :
        a = np.sin(pos/ np.power(10000, ((2 * i) / embedding_dimension)))
        b = np.cos(pos/ np.power(10000, ((2 * i) / embedding_dimension)))
        encoded_pos += [a,b]

    return encoded_pos

In [ ]:
def get_positional_encoding_for_fixed_number_of_words(number_of_words, embedding_dimension):
    encoded_positions = []
    for i in range (number_of_words):
        encoded_positions += [positional_encoder_f(i,embedding_dimension)]

    return np.vstack([encoded_positions])
    
    

layer normalization


In [ ]:
class LayerNorm:
    def __init__ (self, embed_dim):

        self.gamma = np.ones(embed_dim) # initializing to 1 because initially no scaling 
        self.beta = np.zeros(embed_dim) # initializing to 0 because initially no shifting

    def forward (self, x):
        m = np.mean(x, axis = 1, keepdims=True)
        s = np.std(x, axis = 1, keepdims=True)

        normalized = (x - m ) / (s + 1e-5) # incase preventing divison by zero

        return self.gamma * normalized + self.beta

feed forward network


In [ ]:
class FeedForward(nn.Module):
    def __init__(self ,embedding_dimension, hidden_dimension):
        super(FeedForward, self).__init__()
        self.linear1 = nn.Linear(embedding_dimension, hidden_dimension)
        self.linear2 = nn.Linear(hidden_dimension, embedding_dimension)

    def forward(self, x):
        x = self.linear1(x)
        x = torch.relu(x)
        x = self.linear2(x)
        return x

In [ ]:
class EncoderBlock:

    def __init__(self, embed_dim, num_heads, d_ff):
        self.embed_dim = embed_dim
        self.num_heads = num_heads

        self.attention = MultiHeadAttention(self.embed_dim, self.num_heads)

        self.norm = LayerNorm(embed_dim)
        self.norm2 = LayerNorm(embed_dim)
        self.ffn = FeedForward(embed_dim, d_ff )

    

    def forward(self, x):

        # x is with positional encoding information

        attention_output = self.attention.forward(x,x,x) # self attention

        x = x + attention_output

        x = self.norm.forward(x) # learnable beta and gamma

        x_tensor = torch.tensor(x, dtype=torch.float32)

        ff_o = self.ffn.forward(x_tensor)

        x = x + (ff_o).detach().numpy()

        x = self.norm2.forward(x) # learnable beta gamma 2

        return x


In [ ]:
class Embeddings:

    def __init__(self, vocab_size = 100277, embed_dim = 512):

        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        self.embedding_matrix = np.random.randn(vocab_size, embed_dim)
        


    # vocab_size = enc.n_vocab # it has inbuilt special token <|endoftext|>

    def get_embedidngs_with_pe(self, tokens):
        embeddings = []
        token_count = len(tokens)
        for t in tokens:
            embeddings += [self.embedding_matrix[t]]
        embeddings = np.stack(embeddings, axis = 0)
        pe = get_positional_encoding_for_fixed_number_of_words(token_count, self.embed_dim)

        return embeddings + pe
    


### Decoder Block

In [ ]:
class DecoderBlock:

    def __init__(self, embed_dim, num_heads, d_ff):
        
        self.masked_attention = MultiHeadAttention(embed_dim, num_heads, masked = True)

        self.norm1 = LayerNorm(embed_dim)

        self.cross_attention = MultiHeadAttention(embed_dim, num_heads, masked = False)
        self.norm2  = LayerNorm(embed_dim)

        self.ffn = FeedForward(embed_dim, d_ff)
        self.norm3 = LayerNorm(embed_dim)
    

    def forward (self, decoder_input, encoder_output):


        masked_output = self.masked_attention.forward(decoder_input, decoder_input, decoder_input)
        x = decoder_input + masked_output

        # x = self.norm1.forward(x)

        cross_att_op = self.cross_attention.forward(x, encoder_output, encoder_output)
        x = cross_att_op + x

        # x = self.norm2.forward(x)

        x_tensor = torch.tensor(x, dtype=torch.float32)

        ff_o = self.ffn.forward(x_tensor)

        x = x + (ff_o).detach().numpy()
        # x = self.norm3.forward(x)
        return x
        


In [ ]:
class Encoder:

    def __init__(self, num_layers, embed_dim, num_heads, d_ff):

        self.layers  = [EncoderBlock(embed_dim, num_heads, d_ff) for _ in range(num_layers)]

    def forward(self, x):

        for layer in self.layers:
            x = layer.forward(x)

        return x
    
class Decoder:
    def __init__(self,num_layers ,embed_dim, num_heads, d_ff):

        self.layers = [DecoderBlock(embed_dim, num_heads, d_ff) for _ in range(num_layers)]

    def forward (self, x, encoder_output):
        for layer in self.layers:
            x = layer.forward(x, encoder_output)

        return x
    


In [ ]:
class OutputProj(nn.Module):
    def __init__(self, embed_dim, vocab_size):
        super().__init__()

        self.linear = nn.Linear(embed_dim, vocab_size)

    def forward(self, x):
        if not isinstance(x, torch.Tensor):
            x = torch.tensor(x, dtype=torch.float32)
            
        x = self.linear(x)
        return x

In [ ]:
class Transformer:
    def __init__(self, enc_layers, embed_dim, enc_heads, d_enc_ff, dec_layers, dec_heads, d_dec_ff, vocab_size):
        
        self.encoder = Encoder(num_layers = enc_layers, embed_dim = embed_dim, num_heads = enc_heads, d_ff= d_enc_ff )

        self.decoder = Decoder(num_layers = dec_layers, embed_dim = embed_dim, num_heads = dec_heads, d_ff= d_dec_ff )

        self.output_proj = OutputProj(embed_dim=embed_dim, vocab_size= vocab_size)

        self.embedding = Embeddings(vocab_size, embed_dim)

        self.tokenizer = tiktoken.encoding_for_model('gpt-4')\
        
        self.special_tokens = self.tokenizer.special_tokens_set



    #inference
    def forward_inference(self, x): # x is tokens
        x = self.embedding.get_embedidngs_with_pe(x)
        encoder_output = self.encoder.forward(x)
        EOS_TOKEN = 100257
        generated = [EOS_TOKEN]
        K = 10
        MAX_LEN = 100
        for _ in range(MAX_LEN):
            dec_input = self.embedding.get_embedidngs_with_pe(generated)
            decoder_output = self.decoder.forward(dec_input, encoder_output)
            logits = self.output_proj.forward(decoder_output)
            last_logits = logits[-1]
            values, indices = torch.topk(last_logits, K)
            k = torch.randint(0, K, (1,))
            predicted_id = indices[k].item() # random from top k
            if (predicted_id == EOS_TOKEN): # preicted_id is a tensor
                break            
            generated += [predicted_id]
            next_word = self.tokenizer.decode([generated[-1]])
            print(next_word, end = " ")          

In [ ]:
transformer_obj = Transformer( enc_layers = 6, embed_dim =512,enc_heads= 8,d_enc_ff=512, dec_layers = 6,dec_heads = 8,d_dec_ff= 512, vocab_size=100277)

In [ ]:
encoder_input = "The working of transformer "
inputt = transformer_obj.tokenizer.encode(encoder_input)
print(encoder_input, end=" ")
transformer_obj.forward_inference(inputt)


### Output with Layer Normalization in Decoder

The working of transformer  irect  her  Sing  Sing  Evaluate .lang  Evaluate .lang .lang  Sing .lang  her  Sing  Sing  Sing  Evaluate  Sing .lang  Sing  her  Evaluate .lang  Sing  Sing  Evaluate .lang  Sing .lang irect  Sing  Sing  Evaluate .lang  Evaluate  Evaluate  Evaluate  Evaluate  Sing .lang  her  Sing irect  Sing  her  Evaluate  Sing  Sing  her  Sing  her  her  her irect  Sing irect  Sing irect  her  her  Evaluate  Sing  her  her  Sing  Evaluate  Sing .lang  her irect  her  Evaluate  Sing .lang irect  Evaluate irect .lang irect  Evaluate  her .lang  Sing .lang  Sing  Evaluate irect  Evaluate  Sing irect  Sing irect .lang  Sing irect  her irect .lang  Evaluate .lang .lang

### Output without Layer Normalization in Decoder

The working of transformer  myp myp myp myp ап auga si  differing  differing myp ап si _depart ап  libros _cliente myp _cliente errals _cliente myp ап myp myp _cliente myp LEX myp _cliente _cliente errals ап ап ап myp _cliente errals myp ап errals _cliente myp myp LEX myp myp _cliente errals _cliente  Positive  afflict LEX ап _cliente ап ап LEX ап errals myp errals ап _cliente .Constants LEX  reopen  grad ап LEX _UDP  tenía  tolerated  afflict ап .Constants  differing  grad  grad LEX ап ап myp _UDP  grad myp  grad ап  reopen  tenía .Constants errals myp LEX  tolerated myp LEX myp  birthdays errals  tenía